In [ ]:
from pathlib import Path
import pandas as pd
from huggingface_hub import notebook_login
from datasets import Audio, Dataset, Features, Value
NOTEBOOK_DIR = Path.cwd()
CSV_PATH = NOTEBOOK_DIR / "data" / "final_dataset_clean.csv"
AUDIO_DIR = NOTEBOOK_DIR / "audios"
SPK_META_PATH = NOTEBOOK_DIR / "data" / "spk_metadata.csv"
REPO_ID = "mau-cr/mayan-voice"  # <-- cámbialo
PRIVATE = True
SAMPLING_RATE = 16_000

In [ ]:
notebook_login()

In [3]:
df = pd.read_csv(CSV_PATH)
df["audio"] = df["path"].apply(lambda p: str((NOTEBOOK_DIR / p).resolve()))
missing = [p for p in df["audio"] if not Path(p).is_file()]
assert not missing, f"Faltan {len(missing)} audios, ej: {missing[:3]}"
df = df[["audio", "maya", "utt_id", "spk_id"]]
print(df.shape)
df.head()

(2535, 4)


,audio,maya,utt_id,spk_id
0,/home/maucr/Documentos/thesis-mayan-ai/noteboo...,baach,spk_001_utt_0001,spk_001
1,/home/maucr/Documentos/thesis-mayan-ai/noteboo...,chiich,spk_001_utt_0002,spk_001
2,/home/maucr/Documentos/thesis-mayan-ai/noteboo...,ch'íich',spk_001_utt_0003,spk_001
3,/home/maucr/Documentos/thesis-mayan-ai/noteboo...,ja',spk_001_utt_0004,spk_001
4,/home/maucr/Documentos/thesis-mayan-ai/noteboo...,kool,spk_001_utt_0005,spk_001


In [4]:
df.iloc[0]["audio"]

'/home/maucr/Documentos/thesis-mayan-ai/notebooks/create_dataset/audios/spk_001_utt_0001.wav'

In [5]:
from datasets import Dataset, Audio

# Convertir el dataframe a dict y construir desde ahí
ds = Dataset.from_dict({
    "audio":  df["audio"].tolist(),
    "maya":   df["maya"].tolist(),
    "utt_id": df["utt_id"].tolist(),
    "spk_id": df["spk_id"].tolist(),
})

ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
ds

Dataset({
    features: ['audio', 'maya', 'utt_id', 'spk_id'],
    num_rows: 2535
})

In [6]:
import datasets
print(datasets.__version__)

import inspect
print(inspect.signature(Audio.__init__))

4.8.4
(self, sampling_rate: Optional[int] = None, decode: bool = True, num_channels: Optional[int] = None, stream_index: Optional[int] = None, id: Optional[str] = None) -> None


In [7]:
import random
from IPython.display import display, Audio
from datasets import Audio as HFAudio

# Esto es lo que faltaba antes de acceder
ds = ds.cast_column("audio", HFAudio(sampling_rate=16_000))

i = random.randint(0, len(ds)-1)
sample = ds[i]

print({k: v for k, v in sample.items() if k != "audio"})
display(Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"]))

{'maya': "ts'oka'an k beele'ex", 'utt_id': 'spk_006_utt_0301', 'spk_id': 'spk_006'}


In [8]:
sample = ds[0]

print({k: v for k, v in sample.items() if k != "audio"})
print("audio:", sample["audio"]["sampling_rate"], sample["audio"]["array"].shape)

{'maya': 'baach', 'utt_id': 'spk_001_utt_0001', 'spk_id': 'spk_001'}
audio: 16000 (30400,)


In [9]:
ds.push_to_hub(REPO_ID, private=PRIVATE, token=TOKEN)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/2535 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/mau-cr/mayan-voice/commit/00064495bf39253e01eddc4ae5765a121e04e6f3', commit_message='Upload dataset', commit_description='', oid='00064495bf39253e01eddc4ae5765a121e04e6f3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/mau-cr/mayan-voice', endpoint='https://huggingface.co', repo_type='dataset', repo_id='mau-cr/mayan-voice'), pr_revision=None, pr_num=None)